# Schema & Data Exploration — IEPLANE

This notebook connects to the DB2 database and explores every table in the `IEPLANE` schema:
- Row counts & column inventory
- Data types, nulls, and sample values
- Key distributions & date ranges
- Join key validation

Run each cell in order. The output will give you a full picture of the data before building the dashboard.

## 1. Setup & Connection

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

username = os.getenv("DB_USERNAME")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
dbname = os.getenv("DB_NAME")

connection_string = f"db2+ibm_db://{username}:{password}@{host}:{port}/{dbname}"
engine = create_engine(connection_string)

# Helper: run SQL and return DataFrame
def query(sql):
    return pd.read_sql(sql, engine)

print("Connected successfully!")

Connected successfully!


## 2. Discover All Tables in IEPLANE Schema

In [2]:
# List all tables in the IEPLANE schema
tables_df = query("""
    SELECT TABNAME, TYPE, CARD AS ROW_ESTIMATE
    FROM SYSCAT.TABLES
    WHERE TABSCHEMA = 'IEPLANE'
    ORDER BY TABNAME
""")
print(f"Found {len(tables_df)} tables/views in IEPLANE schema:\n")
tables_df.columns = tables_df.columns.str.lower()
tables_df

Found 9 tables/views in IEPLANE schema:



,tabname,type,row_estimate
0,AIRPLANES,T,120
1,AIRPORTS,T,30
2,COUNTRIES,T,196
3,DEPARTMENT,T,22
4,EMPLOYEE,T,1598
5,FLIGHTS,T,297471
6,PASSENGERS,T,50000
7,ROUTES,T,59
8,TICKETS,T,35383337


## 3. Column Inventory for Every Table

In [3]:
# Get columns, data types, and nullability for all tables
columns_df = query("""
    SELECT TABNAME, COLNAME, TYPENAME, LENGTH, SCALE, NULLS, DEFAULT, COLNO
    FROM SYSCAT.COLUMNS
    WHERE TABSCHEMA = 'IEPLANE'
    ORDER BY TABNAME, COLNO
""")

# Normalize column names to lowercase for consistency
columns_df.columns = columns_df.columns.str.lower()

# Display per table
for table_name in sorted(columns_df['tabname'].unique()):
    subset = columns_df[columns_df['tabname'] == table_name]
    print(f"\n{'='*60}")
    print(f"TABLE: IEPLANE.{table_name}  ({len(subset)} columns)")
    print(f"{'='*60}")
    display(subset[['colname', 'typename', 'length', 'scale', 'nulls']].reset_index(drop=True))


TABLE: IEPLANE.AIRPLANES  (13 columns)


,colname,typename,length,scale,nulls
0,AIRCRAFT_REGISTRATION,CHARACTER,7,0,N
1,MODEL,VARCHAR,25,0,N
2,SEATS_BUSINESS,SMALLINT,2,0,Y
3,SEATS_PREMIUM,SMALLINT,2,0,Y
4,SEATS_ECONOMY,SMALLINT,2,0,Y
5,CREW_MEMBERS,SMALLINT,2,0,Y
6,BUILD_DATE,DATE,4,0,N
7,FUEL_GALLONS_HOUR,SMALLINT,2,0,N
8,MAINTENANCE_LAST_ACHECK,DATE,4,0,N
9,MAINTENANCE_LAST_BCHECK,DATE,4,0,N



TABLE: IEPLANE.AIRPORTS  (9 columns)


,colname,typename,length,scale,nulls
0,IATA_CODE,CHARACTER,3,0,N
1,AIRPORT,VARCHAR,50,0,N
2,CITY,VARCHAR,20,0,N
3,COUNTRY,VARCHAR,20,0,N
4,CONTINENT,VARCHAR,20,0,N
5,TIMEZONE,VARCHAR,6,0,N
6,LATITUDE,DOUBLE,8,0,N
7,LONGITUDE,DOUBLE,8,0,N
8,AIRPORT_TAX,DECIMAL,5,2,Y



TABLE: IEPLANE.COUNTRIES  (4 columns)


...

,colname,typename,length,scale,nulls
0,ID,INTEGER,4,0,N
1,FIRSTNME,VARCHAR,20,0,N
2,MIDINIT,CHARACTER,1,0,Y
3,LASTNAME,VARCHAR,20,0,N
4,GENDER,CHARACTER,1,0,Y
5,BIRTH_DATE,DATE,4,0,N
6,PASSPORT,VARCHAR,12,0,N
7,COUNTRY,VARCHAR,50,0,N
8,VIPCARD,VARCHAR,10,0,Y
9,PHONE,CHARACTER,9,0,Y



TABLE: IEPLANE.ROUTES  (7 columns)


,colname,typename,length,scale,nulls
0,ROUTE_CODE,CHARACTER,4,0,N
1,ORIGIN,CHARACTER,3,0,N
2,DESTINATION,CHARACTER,3,0,N
3,PARENT_ROUTE,CHARACTER,6,0,N
4,LEG_NUMBER,SMALLINT,2,0,N
5,DISTANCE,INTEGER,4,0,Y
6,FLIGHT_MINUTES,SMALLINT,2,0,Y



TABLE: IEPLANE.TICKETS  (11 columns)


,colname,typename,length,scale,nulls
0,TICKET_ID,CHARACTER,12,0,N
1,PASSENGER_ID,INTEGER,4,0,N
2,FLIGHT_ID,CHARACTER,6,0,N
3,ROUTE_CODE,CHARACTER,4,0,N
4,DEPARTURE,TIMESTAMP,10,6,N
5,CLASS,CHARACTER,1,0,N
6,SEAT,CHARACTER,4,0,N
7,PRICE,DECIMAL,7,2,N
8,AIRPORT_TAX,DECIMAL,7,2,Y
9,LOCAL_TAX,DECIMAL,7,2,Y


## 4. Row Counts (Actual)

In [4]:
# Normalize column names
tables_df.columns = tables_df.columns.str.lower()

# Get actual row counts for each table
table_names = tables_df[tables_df['type'] == 'T']['tabname'].tolist()

row_counts = []
for t in table_names:
    try:
        count = query(f"SELECT COUNT(*) AS cnt FROM IEPLANE.{t}").iloc[0, 0]
        row_counts.append({'table': t, 'row_count': count})
    except Exception as e:
        row_counts.append({'table': t, 'row_count': f'ERROR: {e}'})

row_counts_df = pd.DataFrame(row_counts)
row_counts_df

,table,row_count
0,AIRPLANES,120
1,AIRPORTS,30
2,COUNTRIES,196
3,DEPARTMENT,22
4,EMPLOYEE,1598
5,FLIGHTS,297471
6,PASSENGERS,50000
7,ROUTES,59
8,TICKETS,35383337


## 5. Sample Data — First 5 Rows of Each Table

In [5]:
# Preview first 5 rows of every table
samples = {}
for t in table_names:
    try:
        df = query(f"SELECT * FROM IEPLANE.{t} FETCH FIRST 5 ROWS ONLY")
        samples[t] = df
        print(f"\n{'='*60}")
        print(f"IEPLANE.{t}  (showing 5 of {row_counts_df.loc[row_counts_df['table']==t, 'row_count'].values[0]} rows)")
        print(f"{'='*60}")
        display(df)
    except Exception as e:
        print(f"\nERROR reading {t}: {e}")


IEPLANE.AIRPLANES  (showing 5 of 120 rows)


,aircraft_registration,model,seats_business,seats_premium,seats_economy,crew_members,build_date,fuel_gallons_hour,maintenance_last_acheck,maintenance_last_bcheck,maintenance_takeoffs,maintenance_flight_hours,total_flight_distance
0,IE00216,AIRBUS A320(320) Express,18,NaN,162,8,2000-01-04,291,2024-05-03,2024-03-08,442,805,612677
1,IE02067,BOEING B737 (800) V3,16,42.0,108,8,2000-01-29,291,2024-03-06,2024-03-09,790,750,198244
2,IE02258,AIRBUS A330-200(332),19,NaN,269,12,2000-02-04,386,2024-04-05,2024-03-15,756,826,598002
3,IE02343,AIRBUS A330-200(332),19,NaN,269,12,2000-02-20,386,2024-04-17,2024-05-18,25,1118,415083
4,IE02478,AIRBUS A320(320) Express,18,NaN,162,8,2000-03-24,291,2024-03-01,2024-04-03,278,286,954376



IEPLANE.AIRPORTS  (showing 5 of 30 rows)


,iata_code,airport,city,country,continent,timezone,latitude,longitude,airport_tax
0,TPA,Tampa International Airport,Tampa,UNITED STATES,AMERICA,CUT-5,27.975500,-82.533203,10.84
1,ATL,Hartsfield Jackson Atlanta,Atlanta,UNITED STATES,AMERICA,CUT-5,33.636700,-84.428101,11.88
2,JFK,John Fitzgerald Kennedy,New York,UNITED STATES,AMERICA,CUT-5,40.639801,-73.778900,NaN
3,LAS,McCarran International Airport,Las Vegas,UNITED STATES,AMERICA,CUT-8,36.080101,-115.152000,11.01
4,TLV,Ben Gurion International Airport,Tel-aviv,ISRAEL,ASIA,CUT+2,32.011398,34.886700,7.78



IEPLANE.COUNTRIES  (showing 5 of 196 rows)


...

,id,firstnme,midinit,lastname,gender,birth_date,passport,country,vipcard,phone,email
0,1,Leland,N,Wicks,M,1967-05-17,332031941,GERMANY,None,110425806,lelan.wicks@gmail.com
1,2,Macie,E,Ballard,F,1982-04-16,337226804,ITALY,None,497857660,None
2,3,Joanna,N,Briggs,F,1964-12-24,927837523,ISRAEL,None,881472718,None
3,4,Angelina,L,Harman,F,1992-04-15,320465558,MEXICO,None,463880963,angelina.harman@gmail.com
4,5,Sevyn,N,Rich,F,2005-10-19,174270897,TAIWAN,None,541052839,None



IEPLANE.ROUTES  (showing 5 of 59 rows)


,route_code,origin,destination,parent_route,leg_number,distance,flight_minutes
0,R002,TPA,ATL,R001,1,655,66
1,R003,ATL,JFK,R001,2,1223,107
2,R006,ATL,JFK,R004,2,1223,107
3,R001,JFK,TPA,R001,0,1621,135
4,R013,JFK,MAD,R007,6,5768,432



IEPLANE.TICKETS  (showing 5 of 35383337 rows)


,ticket_id,passenger_id,flight_id,route_code,departure,class,seat,price,airport_tax,local_tax,total_amount
0,T1234-027643,27336,IE0022,R012,2012-08-25 11:52:00,E,E6,445.96,11.01,93.65,550.62
1,T1234-027644,27347,IE0022,R013,2012-08-25 20:30:00,P,A3,1011.41,0.04,212.39,1223.84
2,T1234-027645,27358,IE0016,R037,2012-08-25 23:05:00,E,K9,935.65,18.71,196.48,1150.84
3,T1234-027646,27366,IE0022,R010,2012-08-25 03:54:00,E,C6,472.26,18.47,99.17,589.90
4,T1234-027647,27366,IE0022,R011,2012-08-25 13:26:00,E,C6,1575.93,16.94,330.94,1923.81


## 6. Primary/Foreign Key Relationships

In [6]:
# Find primary keys
pk_df = query("""
    SELECT T.TABNAME, K.COLNAME, K.COLSEQ
    FROM SYSCAT.TABCONST T
    JOIN SYSCAT.KEYCOLUSE K
        ON T.CONSTNAME = K.CONSTNAME AND T.TABSCHEMA = K.TABSCHEMA
    WHERE T.TABSCHEMA = 'IEPLANE' AND T.TYPE = 'P'
    ORDER BY T.TABNAME, K.COLSEQ
""")
pk_df.columns = pk_df.columns.str.lower()
print("PRIMARY KEYS:")
display(pk_df)

# Find foreign keys
fk_df = query("""
    SELECT
        R.TABNAME AS child_table,
        KC.COLNAME AS child_column,
        R.REFTABNAME AS parent_table,
        KCR.COLNAME AS parent_column
    FROM SYSCAT.REFERENCES R
    JOIN SYSCAT.KEYCOLUSE KC
        ON R.CONSTNAME = KC.CONSTNAME AND R.TABSCHEMA = KC.TABSCHEMA
    JOIN SYSCAT.KEYCOLUSE KCR
        ON R.REFKEYNAME = KCR.CONSTNAME AND R.TABSCHEMA = KCR.TABSCHEMA
        AND KC.COLSEQ = KCR.COLSEQ
    WHERE R.TABSCHEMA = 'IEPLANE'
    ORDER BY R.TABNAME
""")
fk_df.columns = fk_df.columns.str.lower()
print("\nFOREIGN KEYS:")
display(fk_df)

PRIMARY KEYS:


,tabname,colname,colseq
0,AIRPLANES,AIRCRAFT_REGISTRATION,1
1,AIRPORTS,IATA_CODE,1
2,DEPARTMENT,DEPTNO,1
3,EMPLOYEE,EMPNO,1
4,FLIGHTS,FLIGHT_ID,1
5,FLIGHTS,ROUTE_CODE,2
6,FLIGHTS,DEPARTURE,3
7,PASSENGERS,ID,1
8,ROUTES,ROUTE_CODE,1
9,TICKETS,TICKET_ID,1



FOREIGN KEYS:


,child_table,child_column,parent_table,parent_column
0,EMPLOYEE,WORKDEPT,DEPARTMENT,DEPTNO
1,FLIGHTS,ROUTE_CODE,ROUTES,ROUTE_CODE
2,FLIGHTS,AIRPLANE,AIRPLANES,AIRCRAFT_REGISTRATION
3,ROUTES,ORIGIN,AIRPORTS,IATA_CODE
4,ROUTES,DESTINATION,AIRPORTS,IATA_CODE
5,ROUTES,PARENT_ROUTE,ROUTES,ROUTE_CODE
6,TICKETS,FLIGHT_ID,FLIGHTS,FLIGHT_ID
7,TICKETS,ROUTE_CODE,FLIGHTS,ROUTE_CODE
8,TICKETS,DEPARTURE,FLIGHTS,DEPARTURE
9,TICKETS,PASSENGER_ID,PASSENGERS,ID


## 7. Null Analysis — Missing Data per Column

In [ ]:
# Check null counts for each table
for t in table_names:
    try:
        cols = columns_df[columns_df['tabname'] == t]['colname'].tolist()
        if not cols:
            # Try uppercase match in case tabname values are uppercase
            cols = columns_df[columns_df['tabname'] == t.upper()]['colname'].tolist()
        null_parts = [f'SUM(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END) AS "{c}"' for c in cols]
        sql = f"SELECT {', '.join(null_parts)} FROM IEPLANE.{t}"
        null_df = query(sql)
        
        total = row_counts_df.loc[row_counts_df['table']==t, 'row_count'].values[0]
        null_summary = null_df.T.rename(columns={0: 'null_count'})
        null_summary['pct_null'] = (null_summary['null_count'] / total * 100).round(2)
        null_summary = null_summary[null_summary['null_count'] > 0]
        
        if len(null_summary) > 0:
            print(f"\n{t} — columns with NULLs:")
            display(null_summary)
        else:
            print(f"\n{t} — no NULLs found ✓")
    except Exception as e:
        print(f"\nERROR on {t}: {e}")


AIRPLANES — columns with NULLs:


,null_count,pct_null
seats_business,20,16.67
seats_premium,79,65.83
seats_economy,11,9.17



AIRPORTS — columns with NULLs:


,null_count,pct_null
airport_tax,8,26.67



COUNTRIES — columns with NULLs:


,null_count,pct_null
capital,5,2.55
population,13,6.63



DEPARTMENT — no NULLs found ✓

EMPLOYEE — columns with NULLs:


,null_count,pct_null
is_external,79,4.94
phoneno,167,10.45
is_parttime,157,9.82
salary,71,4.44
bonus,200,12.52
comm,1311,82.04



FLIGHTS — no NULLs found ✓

PASSENGERS — columns with NULLs:


,null_count,pct_null
vipcard,39887,79.77
phone,15088,30.18
email,15015,30.03


## 8. Key Metrics Exploration

### 8a. FLIGHTS — Date Range & Volume

In [ ]:
query("""
    SELECT 
        COUNT(*) AS total_flights,
        COUNT(DISTINCT flight_id) AS distinct_flight_ids,
        MIN(departure) AS earliest_departure,
        MAX(departure) AS latest_departure,
        COUNT(DISTINCT airplane) AS distinct_airplanes,
        COUNT(DISTINCT route_code) AS distinct_routes,
        AVG(price_economy) AS avg_price_economy,
        AVG(price_business) AS avg_price_business
    FROM IEPLANE.FLIGHTS
""")

### 8b. TICKETS — Revenue Overview

In [ ]:
query("""
    SELECT 
        COUNT(*) AS total_tickets,
        SUM(total_amount) AS total_revenue,
        AVG(total_amount) AS avg_ticket_price,
        MIN(total_amount) AS min_ticket,
        MAX(total_amount) AS max_ticket
    FROM IEPLANE.TICKETS
""")

### 8c. AIRPLANES — Fleet Overview

In [ ]:
query("""
    SELECT 
        COUNT(*) AS total_aircraft,
        COUNT(DISTINCT model) AS distinct_models,
        AVG(total_flight_distance) AS avg_flight_distance,
        AVG(flight_hours) AS avg_flight_hours,
        AVG(fuel_gallons_hour) AS avg_fuel_gph
    FROM IEPLANE.AIRPLANES
""")

### 8d. PASSENGERS — Demographics

In [ ]:
query("""
    SELECT 
        COUNT(*) AS total_passengers,
        COUNT(DISTINCT gender) AS gender_categories,
        AVG(age) AS avg_age,
        MIN(age) AS min_age,
        MAX(age) AS max_age
    FROM IEPLANE.PASSENGERS
""")

### 8e. AIRPORTS — Network

In [ ]:
query("""
    SELECT COUNT(*) AS total_airports
    FROM IEPLANE.AIRPORTS
""")

### 8f. EMPLOYEES & DEPARTMENTS

In [ ]:
# Departments
print("DEPARTMENTS:")
display(query("SELECT * FROM IEPLANE.DEPARTMENT"))

# Employee summary
print("\nEMPLOYEE SUMMARY BY DEPARTMENT:")
display(query("""
    SELECT 
        d.department_name,
        COUNT(*) AS headcount,
        AVG(e.salary) AS avg_salary,
        SUM(e.salary) AS total_salary_cost
    FROM IEPLANE.EMPLOYEE e
    LEFT JOIN IEPLANE.DEPARTMENT d ON e.department_id = d.department_id
    GROUP BY d.department_name
    ORDER BY headcount DESC
"""))

## 9. Join Validation — Can We Calculate Load Factor?

Load Factor = Tickets Sold / Available Seats per Flight

This requires joining TICKETS → FLIGHTS → AIRPLANES (for seat capacity).

In [ ]:
# Check: How do TICKETS link to FLIGHTS?
print("TICKETS sample (check for flight reference column):")
display(query("SELECT * FROM IEPLANE.TICKETS FETCH FIRST 3 ROWS ONLY"))

print("\nFLIGHTS sample (check airplane column):")
display(query("SELECT flight_id, airplane, route_code FROM IEPLANE.FLIGHTS FETCH FIRST 3 ROWS ONLY"))

print("\nAIRPLANES sample (check capacity columns):")
display(query("SELECT * FROM IEPLANE.AIRPLANES FETCH FIRST 3 ROWS ONLY"))

## 10. ROUTES — Understanding the Network

In [ ]:
print("ROUTES sample:")
display(query("SELECT * FROM IEPLANE.ROUTES FETCH FIRST 10 ROWS ONLY"))

print("\nTotal routes:")
display(query("SELECT COUNT(*) AS total_routes, COUNT(DISTINCT route_code) AS distinct_codes FROM IEPLANE.ROUTES"))

## 11. Summary of Findings

After running all cells, fill in the key findings here:

| Item | Value |
|------|-------|
| Total Tables | |
| Date Range | |
| Total Flights | |
| Total Tickets / Revenue | |
| Fleet Size / Models | |
| Airports in Network | |
| Total Passengers | |
| Employees | |
| Key Join: Tickets → Flights | |
| Key Join: Flights → Airplanes | |
| Notable Data Issues | |